# modernBERT Multi-Task CI/CD Log Classifier (REAL-ONLY)

Fine-tune **answerdotai/ModernBERT-large** to predict **Category + Component** from a log line.

> Run this notebook on a **Kaggle GPU T4** accelerator. No local training required.
> Training data: **`train_real.csv`** — real log lines only (original researcher-labeled rows + Veloren
> real GitLab CI failure logs). No synthetic rows, no heuristic/pseudo labels.
> Severity is **not** trained here: the small model predicts category + component, and severity is
> classified at predict time by a free LLM (Google Gemini) with a deterministic policy fallback
> (see the `severity_llm` notebook). Model saved to `/kaggle/working/modernbert-multitask`.

In [ ]:
# 1. Install dependencies (run once)
%pip install -q "transformers==5.17.0" "tokenizers==0.23.2" "datasets==5.0.1" "accelerate==1.15.0" "evaluate==0.4.6"
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
print("done")

In [ ]:
# 2. Config
import os

MODEL_ID = "answerdotai/ModernBERT-large"   # modernBERT large (best accuracy); heads: category + component
MAX_LENGTH = 256
BATCH_SIZE = 8                              # T4 16GB fits batch 8 with grad accum
GRAD_ACCUM = 4                              # effective batch 32
EPOCHS = 8
LR = 2e-5
OUTPUT_DIR = "/kaggle/working/modernbert-multitask"
HUB_REPO = None                             # e.g. "azizmadhbouh/cicd-log-multitask" or None
KAGGLE_DATASET = "/kaggle/input/cicd-selfhealing-dataset"  # upload pipeline-dataset/train_real.csv (find_csvs globs any CSV)

# --- (disabled) LogChunks pseudo-labeling fine-tune --- real-only policy: would require heuristic labels
USE_LOGCHUNKS = False
LOGCHUNKS_URL = "https://zenodo.org/api/records/3632351/files/LogChunks.zip/content"
LOGCHUNKS_ZIP = "/kaggle/working/LogChunks.zip"
LOGCHUNKS_MAX_CHUNK = 5000        # drop failure chunks longer than this (log noise)
PSEUDO_MIN_CONF = 0.85            # min softmax confidence across all predicted heads to accept a pseudo-label
PSEUDO_EPOCHS = 2                 # continued fine-tune epochs on pseudo-labeled real logs
PSEUDO_LR = 5e-6                  # lower LR for the fine-tune phase

# --- FlakeStorm (full set gated/private; Veloren subset already merged into train_real.csv) ---
USE_FLAKESTORM = False             # set True to download ~4.2k real CI failure logs from HF
FLAKESTORM_REPO = "ahenrij/flakestorm"

# If ml/logchunks_pseudo_label_config.json was uploaded with the dataset, load it (overrides above).
import json, glob
_cfgs = glob.glob("/kaggle/input/**/logchunks_pseudo_label_config.json", recursive=True)
if _cfgs:
    _cfg = json.load(open(_cfgs[0], encoding="utf-8"))
    USE_LOGCHUNKS = _cfg.get('enable', USE_LOGCHUNKS)
    LOGCHUNKS_URL = _cfg.get('source', {}).get('url', LOGCHUNKS_URL)
    LOGCHUNKS_ZIP = _cfg.get('source', {}).get('zip_path', LOGCHUNKS_ZIP)
    LOGCHUNKS_MAX_CHUNK = _cfg.get('source', {}).get('max_chunk_len', LOGCHUNKS_MAX_CHUNK)
    PSEUDO_MIN_CONF = _cfg.get('classification', {}).get('min_confidence', PSEUDO_MIN_CONF)
    PSEUDO_EPOCHS = _cfg.get('fine_tune', {}).get('epochs', PSEUDO_EPOCHS)
    PSEUDO_LR = _cfg.get('fine_tune', {}).get('lr', PSEUDO_LR)
    print("loaded LogChunks pseudo-label config from", _cfgs[0])
print("config set")

In [ ]:
# 3. Locate the dataset CSVs (train/val/test)
import glob, os

def find_csvs():
    # exact mount first, then recursive search anywhere under /kaggle/input
    cand = set()
    if os.path.isdir(KAGGLE_DATASET):
        cand |= set(glob.glob(os.path.join(KAGGLE_DATASET, "*.csv")))
    cand |= set(glob.glob("/kaggle/input/**/*.csv", recursive=True))
    named = {}
    for p in cand:
        b = os.path.basename(p).lower()
        if b in ("train.csv", "val.csv", "test.csv") and b not in named:
            named[b] = p
    if all(k in named for k in ("train.csv", "val.csv", "test.csv")):
        return named
    if cand:
        return {"source": sorted(cand)}   # single-file dataset -> auto-split in cell 4
    raise FileNotFoundError("No CSV found under /kaggle/input - attach the dataset in the Kaggle UI (Add Input).")

paths = find_csvs()
print("found:", paths)

In [ ]:
# 3b. Dataset provenance (read-only note)
# Real-only policy: train_real.csv contains ONLY real log lines - original researcher-labeled rows
# plus the Veloren real GitLab CI failure logs (labels derived from FlakeStorm root-cause classes).
# No synthetic rows and no heuristic/pseudo-labeled rows. The FlakeStorm full set is gated/private;
# the reachable Veloren part is already merged into train_real.csv, so nothing extra is downloaded.
print('provenance: real-only (no synthetic, no heuristic labels)')

In [ ]:
# 4. Load CSVs + build label maps (category, component)
import pandas as pd
import json
from sklearn.model_selection import train_test_split

def load_csv(p):
    df = pd.read_csv(p, encoding="utf-8")
    df = df[["error_message", "category", "component"]].dropna()
    df = df[df["error_message"].str.strip() != ""]
    return df

def stratify_split(df):
    # strategy: full (category,component) when big enough, else category only
    try:
        s = df["category"] + "|" + df["component"]
        train, rest = train_test_split(df, test_size=0.3, stratify=s, random_state=42)
        s2 = rest["category"] + "|" + rest["component"]
        val, test = train_test_split(rest, test_size=0.5, stratify=s2, random_state=42)
    except ValueError:
        s = df["category"]
        train, rest = train_test_split(df, test_size=0.3, stratify=s, random_state=42)
        s2 = rest["category"]
        val, test = train_test_split(rest, test_size=0.5, stratify=s2, random_state=42)
    return (train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True))

if "source" in paths:
    print("No train/val/test split found - auto-splitting", len(paths["source"]), "file(s)")
    df = pd.concat([load_csv(p) for p in paths["source"]], ignore_index=True)
    df = df.drop_duplicates(subset=["error_message"])
    tr, va, te = stratify_split(df)
    splits = {"train": tr, "val": va, "test": te}
else:
    splits = {}
    for name, p in paths.items():
        key = os.path.splitext(name)[0]
        splits[key] = load_csv(p)
print({k: len(v) for k, v in splits.items()})

label_maps = {}
for col in ["category", "component"]:
    values = sorted({v for df in splits.values() for v in df[col].unique()})
    label_maps[col] = {v: i for i, v in enumerate(values)}
    print(col, "->", len(values), "labels")

os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(os.path.join(OUTPUT_DIR, "label_maps.json"), "w", encoding="utf-8") as f:
    json.dump(label_maps, f, indent=2)
num_labels = {col: len(label_maps[col]) for col in label_maps}
print("num_labels:", num_labels)

In [ ]:
# 5. Multi-task modernBERT model (2 heads: category/component; severity is LLM-classified at predict time)
import torch
import torch.nn as nn
from transformers import ModernBertConfig, ModernBertModel, PreTrainedModel
from transformers.modeling_outputs import ModelOutput

TASKS = ("category", "component")

class ModernBERTMultiTask(PreTrainedModel):
    config_class = ModernBertConfig
    base_model_prefix = "modernbert"
    main_input_name = "input_ids"
    supports_gradient_checkpointing = True
    _supports_sdpa = True
    _supports_flash_attn = False

    def __init__(self, config, num_labels=None):
        config._attn_implementation = "eager"
        config.reference_compile = False  # avoid torch.compile auto-enable (crashes in DataParallel/post_init)
        super().__init__(config)
        if num_labels is None:
            num_labels = getattr(config, "multitask_num_labels", {})
        self.num_labels = num_labels
        self._weight_buffers = {}
        for task in TASKS:
            w = (getattr(config, "task_class_weights", None) or {}).get(task)
            if w is not None:
                buf = torch.tensor(w, dtype=torch.float32)
                self._weight_buffers[task] = buf
                self.register_buffer(f"task_weights_{task}", buf)
        self.modernbert = ModernBertModel(config)
        self.heads = nn.ModuleDict(
            {task: nn.Linear(config.hidden_size, num_labels[task]) for task in TASKS}
        )
        self.post_init()

    @classmethod
    def from_pretrained(cls, *args, **kwargs):
        kwargs.setdefault("torch_dtype", torch.float32)
        kwargs.setdefault("trust_remote_code", True)
        kwargs.setdefault("attn_implementation", "eager")
        model = super().from_pretrained(*args, **kwargs)
        for name, buf in model.named_buffers():
            if buf.is_meta:
                setattr(model, name, torch.empty(buf.shape, dtype=buf.dtype))
        model._weight_buffers = {
            t: getattr(model, f"task_weights_{t}")
            for t in TASKS if hasattr(model, f"task_weights_{t}")
        }
        return model

    def forward(self, input_ids=None, attention_mask=None, labels=None, return_dict=None, **kwargs):
        outputs = self.modernbert(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        pooled = outputs.last_hidden_state[:, 0]
        logits = {task: self.heads[task](pooled) for task in TASKS}
        loss = None
        if labels is not None:
            losses = []
            for t in TASKS:
                w = self._weight_buffers.get(t)
                losses.append(nn.functional.cross_entropy(
                    logits[t], labels[t].long().view(-1),
                    weight=None if w is None else w.to(logits[t].device),
                ))
            loss = torch.stack(losses).mean()
        return ModelOutput(loss=loss, logits=logits)

# Load base weights into the wrapper (keeps heads randomly initialised)
print("Loading base model...")
_base_cfg = ModernBertConfig.from_pretrained(MODEL_ID)
_base_cfg.reference_compile = False
base = ModernBertModel.from_pretrained(MODEL_ID, trust_remote_code=True, config=_base_cfg)
model = ModernBERTMultiTask(base.config, num_labels)
model.modernbert = base
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Loaded ModernBERTMultiTask: {n_params:.1f}M params")

# inverse-frequency class weights for component head
def inv_freq_weights(labels, count):
    n = len(labels); tot = int(count.sum())
    w = [1.0] * n
    for lab, cnt in count.items():
        w[labels[lab]] = tot / (n * int(cnt))
    w = torch.tensor(w, dtype=torch.float32)
    return w / w.mean()

for wtask in ("component",):
    w_arr = inv_freq_weights(label_maps[wtask], splits["train"][wtask].value_counts())
    model.register_buffer(f"task_weights_{wtask}", w_arr)
    model._weight_buffers[wtask] = w_arr
    print(f"{wtask} weights:", dict(zip(label_maps[wtask], [round(float(x), 2) for x in w_arr])))

In [ ]:
# 6. Build HF datasets + tokenize
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

def make_ds(df):
    ds = Dataset.from_pandas(df)
    return ds

ds = DatasetDict({name: make_ds(df) for name, df in splits.items()})

def tok(batch):
    return tokenizer(batch["error_message"], truncation=True, padding=False, max_length=MAX_LENGTH)

ds = ds.map(tok, batched=True)

label_cols = [f"{t}_id" for t in TASKS]
def encode(batch):
    return {f"{t}_id": [label_maps[t][v] for v in batch[t]] for t in TASKS}

ds = ds.map(encode, batched=True)
for name in ds:
    ds[name].set_format("torch", columns=["input_ids", "attention_mask"] + label_cols)
print({k: len(ds[k]) for k in ds})

In [ ]:
# 7. Collator + metrics
import numpy as np
from transformers.trainer_utils import EvalPrediction

def collate(batch):
    padded = tokenizer.pad(
        {"input_ids": [b["input_ids"].tolist() for b in batch],
         "attention_mask": [b["attention_mask"].tolist() for b in batch]},
        padding=True,
        return_tensors="pt",
    )
    labels = {t: torch.stack([b[f"{t}_id"] for b in batch]) for t in TASKS}
    return {"input_ids": padded["input_ids"], "attention_mask": padded["attention_mask"], "labels": labels}

def compute_metrics(p: EvalPrediction):
    res = {}
    for t in TASKS:
        preds = np.argmax(p.predictions[t], axis=-1)
        refs = p.label_ids[t]
        res[f"{t}_acc"] = float(np.mean(preds == refs))
    res["avg_acc"] = float(np.mean([res[f"{t}_acc"] for t in TASKS]))
    return res

print("ok")

In [ ]:
# 8. TrainingArguments (T4 optimized: fp16, grad accum, grad checkpointing)
import torch
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=50,
    lr_scheduler_type="linear",
    fp16=torch.cuda.is_available(),
    bf16=False,
    optim="adamw_torch",
    weight_decay=0.01,
    gradient_checkpointing=True,
    save_strategy="steps",
    save_steps=1000,                  # only ~1360 total steps, so only 1 intermediate save
    save_total_limit=2,               # keep only 2 checkpoints max (avoids filling 20GB disk)
    eval_strategy="steps",
    eval_steps=250,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="avg_acc",
    greater_is_better=True,
    remove_unused_columns=False,
    report_to=[],
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds.get("val", ds["train"]),
    data_collator=collate,
    compute_metrics=compute_metrics,
)
print("trainer ready, device:", next(model.parameters()).device)

In [ ]:
# 9. TRAIN
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("Starting training on", device_name)
trainer.train()
print("training done (best checkpoint reloaded by load_best_model_at_end)")


In [ ]:
# 9b. LogChunks real-world augmentation + final save
import urllib.request, zipfile, xml.etree.ElementTree as ET, collections

def load_logchunks(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        rows = []
        for name in z.namelist():
            norm = name.replace("\\", "/")
            if "/build-failure-reason/" not in norm or not norm.endswith(".xml"):
                continue
            if norm.startswith("__MACOSX"):
                continue
            root = ET.fromstring(z.read(name))
            for ex in root.findall("Example"):
                chunk = (ex.findtext("Chunk") or "").strip()
                log = (ex.findtext("Log") or "").strip()
                kw = (ex.findtext("Keywords") or "").strip()
                cat = (ex.findtext("Category") or "0").strip()
                if not chunk:
                    continue
                rows.append({"error_message": chunk, "keywords": kw,
                             "struct_category": cat, "repo": "/".join(log.split("/")[:2])})
        return pd.DataFrame(rows)

def predict_with_conf(model, tokenizer, texts, batch_size=16):
    device = next(model.parameters()).device
    model.eval()
    preds = {t: [] for t in TASKS}
    confs = {t: [] for t in TASKS}
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i+batch_size], truncation=True, padding=True,
                        max_length=MAX_LENGTH, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        for t in TASKS:
            prob = torch.softmax(logits[t], dim=-1)
            conf, idx = torch.max(prob, dim=-1)
            preds[t].extend(idx.tolist())
            confs[t].extend(conf.tolist())
    return preds, confs

def cap_by_train_counts(pool, caps):
    used = {t: collections.Counter() for t in caps}
    keep = []
    for _, row in pool.iterrows():
        ok = True
        for t in caps:
            lab = row[f"{t}_pred"]
            if used[t][lab] >= caps[t].get(lab, 0):
                ok = False
                break
        if ok:
            for t in caps:
                used[t][row[f"{t}_pred"]] += 1
            keep.append(row)
    return pd.DataFrame(keep)

aug_count = 0
try:
    if USE_LOGCHUNKS:
        if not os.path.exists(LOGCHUNKS_ZIP):
            print("Downloading LogChunks (~24MB) from Zenodo ...")
            urllib.request.urlretrieve(LOGCHUNKS_URL, LOGCHUNKS_ZIP)
        ldf = load_logchunks(LOGCHUNKS_ZIP)
        ldf = ldf[ldf["error_message"].str.len() <= LOGCHUNKS_MAX_CHUNK]
        print(f"Parsed {len(ldf)} LogChunks failure chunks from {ldf['repo'].nunique()} repos")
        inv = {t: {i: v for v, i in label_maps[t].items()} for t in TASKS}
        preds, confs = predict_with_conf(model, tokenizer, ldf["error_message"].tolist())
        for t in TASKS:
            ldf[f"{t}_pred"] = [inv[t][p] for p in preds[t]]
            ldf[f"{t}_conf"] = confs[t]
        ldf["min_conf"] = ldf[[f"{t}_conf" for t in TASKS]].min(axis=1)
        pool = ldf[ldf["min_conf"] >= PSEUDO_MIN_CONF].drop_duplicates(subset="error_message").copy()
        print(f"high-confidence pool: {len(pool)}")
        caps = {t: splits["train"][t].value_counts().to_dict() for t in TASKS}
        pool = cap_by_train_counts(pool, caps)
        print(f"after train-distribution cap: {len(pool)}")
        if len(pool):
            aug_df = pd.DataFrame({
                "error_message": pool["error_message"].to_list(),
                "category": pool["category_pred"].to_list(),
                "component": pool["component_pred"].to_list(),
            })
            merged = pd.concat([splits["train"].reset_index(drop=True), aug_df], ignore_index=True)
            comb = Dataset.from_pandas(merged)
            comb = comb.map(tok, batched=True).map(encode, batched=True)
            comb.set_format("torch", columns=["input_ids", "attention_mask"] + label_cols)
            ft_args = TrainingArguments(
                output_dir=OUTPUT_DIR + "-ft",
                num_train_epochs=PSEUDO_EPOCHS,
                per_device_train_batch_size=BATCH_SIZE,
                per_device_eval_batch_size=BATCH_SIZE,
                gradient_accumulation_steps=GRAD_ACCUM,
                learning_rate=PSEUDO_LR,
                warmup_steps=0,
                lr_scheduler_type="linear",
                fp16=torch.cuda.is_available(),
                bf16=False,
                optim="adamw_torch",
                weight_decay=0.01,
                gradient_checkpointing=True,
                save_strategy="no",
                logging_steps=5,
                report_to=[],
                seed=42,
            )
            ft_trainer = Trainer(model=model, args=ft_args, train_dataset=comb, data_collator=collate)
            print(f"Fine-tuning on {len(aug_df)} pseudo-labeled real CI logs ...")
            ft_trainer.train()
            aug_count = len(aug_df)
        else:
            print("No LogChunks chunks above threshold - skipping fine-tune")
    else:
        print("USE_LOGCHUNKS=False - skipping augmentation")
except Exception as e:
    print("[warn] LogChunks augmentation skipped:", repr(e))
    aug_count = 0

# Final save (after all training) - this is the model pushed to the Hub
model.config.multitask_num_labels = model.num_labels
if model._weight_buffers:
    model.config.task_class_weights = {t: w.tolist() for t, w in model._weight_buffers.items()}
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
with open(os.path.join(OUTPUT_DIR, "label_maps.json"), "w", encoding="utf-8") as f:
    json.dump(label_maps, f, indent=2)
print(f"final model saved to {OUTPUT_DIR} (pseudo-labeled real logs used: {aug_count})")

# IMPORTANT: /kaggle/working is DELETED when this session ends.
# Download the 'modernbert-multitask' folder NOW from the Kaggle output panel (right-click → Download).
# For a permanent backup, set HUB_REPO in the config cell above and re-run — the model will push to HuggingFace Hub.

In [ ]:
# 10. Evaluate on held-out test set
import json
test_res = trainer.evaluate(eval_dataset=ds["test"])
print("=== TEST METRICS ===")
for k in sorted(test_res):
    v = test_res[k]
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
print()
if HUB_REPO:
    print("will push to", HUB_REPO)

In [ ]:
# 11. (Optional) Push to HuggingFace Hub
if HUB_REPO:
    from huggingface_hub import login
    import getpass
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = getpass.getpass("HuggingFace write token: ")
    login(token=os.environ["HF_TOKEN"])
    model.push_to_hub(HUB_REPO)
    tokenizer.push_to_hub(HUB_REPO)
    from huggingface_hub import upload_file
    upload_file(path_or_fileobj=OUTPUT_DIR + "/label_maps.json",
                path_in_repo="label_maps.json", repo_id=HUB_REPO)
    print("pushed to https://huggingface.co/" + HUB_REPO)
else:
    print("HUB_REPO not set - skipping push")

In [ ]:
# 12. Quick sanity: classify a few real logs
from transformers import AutoTokenizer
import torch

model.eval()
example_logs = [
    "[INFRA] time=\"2024-05-06T03:44:00Z\" level=warning msg=\"x509: certificate is not yet valid; clock skew detected: delta=245s\" host=ci-node-10",
    "ERROR: Cannot connect to the Docker daemon at unix:///var/run/docker.sock. Is the docker daemon running?",
    "ModuleNotFoundError: No module named 'psycopg2'",
    "FAILED tests/test_api.py::test_auth - AssertionError: 403 != 200",
]
inv = {t: {i: v for v, i in label_maps[t].items()} for t in TASKS}
enc = tokenizer(example_logs, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
enc = {k: v.to(next(model.parameters()).device) for k, v in enc.items()}
with torch.no_grad():
    logits = model(**enc).logits
for i, text in enumerate(example_logs):
    print("---", text[:80])
    for t in TASKS:
        pred = int(torch.argmax(logits[t][i]))
        print(f"   {t}: {inv[t][pred]}")

## Next steps
1. The trained model is in `/kaggle/working/modernbert-multitask` (download via the notebook output panel).
2. Set `HUB_REPO` above and push to HF Hub, then `predict.py` loads it from the Hub (category + component).
3. Severity is produced at predict time by the free Google Gemini classifier (`severity_llm` notebook),
   with a deterministic policy fallback for offline/rate-limit cases.
4. To re-validate on fresh real logs, run `predict.py` against `jenkins_output_6/7/8.txt` (sequentially).